In [3]:
from gitsource import GithubRepositoryDataReader
from openai import OpenAI
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
from dotenv import load_dotenv
load_dotenv()

files = reader.read()
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)
# print(len(documents))
total_documents = len(documents)
# print(f"Total lesson pages: {total_documents}")

from minsearch import Index
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "content": 0.5},
    num_results=1
)

[doc["filename"] for doc in search_results]

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(doc["filename"])
        lines.append("content: " + doc["content"])
        lines.append("")

    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()
prompt = build_prompt(question, search_results)

# print(prompt)
openai_client = OpenAI()
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

print(response.usage.input_tokens)
print(response.output_text)


2290
194
The agentic loop keeps calling the model with a `while True` loop, and it stops only when the model returns **no function calls**.

### How it works
1. Send the current `messages` to the model.
2. Append the model’s output to the conversation history.
3. If the model produced a `function_call`:
   - run the tool
   - append the tool result back into `messages`
   - set `has_function_calls = True`
4. If there were **no** function calls in that response:
   - `break` out of the loop

### The key stop condition
```python
if has_function_calls == False:
    break
```

So the model controls how many searches/tools happen, and your code just keeps looping until the model gives a final answer without requesting any more tools.

If you want, I can also show this as a simple flowchart or pseudocode.
